[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/ray-certified/notebooks/day-07-ray-serve-deployments.ipynb#scrollTo=a1b2c3d4)

---
# Day 7 · Ray Serve — Model Serving, Deployments, and HTTP Endpoints
**certified-journeys / ray-certified** · Practice Day

> **Goal for today:** Define, configure, compose, and query Ray Serve deployments — including an sklearn model served over HTTP and a chained Preprocessor → Predictor pipeline.


In [ ]:
%pip install -q 'ray[serve]' scikit-learn requests fastapi


## Step 1 · What is Ray Serve?

Ray Serve is a scalable model-serving library built on top of Ray. Key ideas:

| Concept | Meaning |
|---|---|
| **Deployment** | A class or function annotated with `@serve.deployment` |
| **Replica** | One actor instance of a deployment; multiple replicas = load balanced |
| **Ingress** | The deployment that receives HTTP traffic (usually the first one) |
| **Handle** | An async ref used to call one deployment from another |
| **serve.run()** | Deploys the application graph and returns a handle |

Serve decouples **scaling** from **logic**: you tune `num_replicas` and `ray_actor_options` without changing application code.


In [ ]:
import ray
from ray import serve
import requests

# Start Ray and Ray Serve in the current process
# In a notebook, `ray.init()` without arguments uses a local cluster.
ray.init(ignore_reinit_error=True)
serve.start(detached=False)  # detached=False → Serve shuts down when script exits

# ── Minimal deployment ──────────────────────────────────────────────────────
@serve.deployment
class HelloWorld:
    """The simplest possible Serve deployment."""

    async def __call__(self, request):
        """Serve calls __call__ for every incoming HTTP request."""
        name = (await request.json()).get("name", "World")
        return {"message": f"Hello, {name}!"}

# bind() creates an application node; serve.run() deploys it
handle = serve.run(HelloWorld.bind(), name="hello-app", route_prefix="/hello")
print("Deployment handle:", handle)


### What just happened?
- **`@serve.deployment`** registers the class with Serve's runtime — it does not instantiate anything yet.
- **`.bind()`** creates an immutable application description (a DAG node); think of it as "snapshot of config + class".
- **`serve.run()`** materialises the deployment: spawns replica actors, registers the route, returns an in-process handle.
- **`detached=False`** means the Serve controller lives only while the Python process is alive — perfect for notebooks.


In [ ]:
# Query the deployment over HTTP (Serve starts an HTTP server on port 8000 by default)
import time
time.sleep(1)  # brief pause to let replicas finish starting

resp = requests.post("http://localhost:8000/hello", json={"name": "Ray Learner"})
print("Status:", resp.status_code)
print("Body  :", resp.json())

# You can also call it in-process via the async handle (useful in tests)
import asyncio
result = asyncio.get_event_loop().run_until_complete(
    handle.remote("dummy_request")   # handle.remote() returns an ObjectRef
)
# Note: handle.remote() bypasses HTTP and calls the deployment actor directly
print("In-process handle call:", result)


### What just happened?
- **`requests.post`** travels through Serve's built-in HTTP server (Uvicorn), which routes to a replica actor.
- **`handle.remote()`** is the in-process shortcut — skips HTTP, calls the actor directly via Ray's IPC.
- Both paths hit the same `__call__` method; the difference is transport, not logic.


## Step 2 · Deployment Configuration: replicas, resources, autoscaling

The `@serve.deployment` decorator accepts configuration kwargs:

| Parameter | Effect |
|---|---|
| `num_replicas` | Static number of actor instances (mutually exclusive with autoscaling) |
| `ray_actor_options` | Resources per replica — CPUs, GPUs, memory |
| `autoscaling_config` | Dict with `min_replicas`, `max_replicas`, `target_num_ongoing_requests_per_replica` |
| `max_concurrent_queries` | Max in-flight requests per replica (backpressure valve) |

Tip: for large models where loading is expensive, prefer `max_concurrent_queries` over many replicas.


In [ ]:
# Deployment with static replica count and resource reservation
@serve.deployment(
    num_replicas=2,                          # 2 actor instances behind the load balancer
    ray_actor_options={"num_cpus": 0.5},     # each replica claims 0.5 CPU (Colab-friendly)
    max_concurrent_queries=5,               # each replica handles up to 5 requests at once
)
class EchoService:
    def __init__(self):
        import os
        self.pid = os.getpid()  # track which replica handled the request

    async def __call__(self, request):
        body = await request.json()
        return {"echo": body, "replica_pid": self.pid}

echo_handle = serve.run(
    EchoService.bind(),
    name="echo-app",
    route_prefix="/echo",
)
time.sleep(1)

# Send 4 requests — you may see different PIDs (different replicas)
for i in range(4):
    r = requests.post("http://localhost:8000/echo", json={"value": i})
    print(f"Request {i}: replica_pid={r.json()['replica_pid']}")


### What just happened?
- **Two replicas** means two actor processes; Serve round-robins requests across them.
- **`ray_actor_options`** is passed directly to `ray.remote()` under the hood — same syntax as regular remote functions.
- **`max_concurrent_queries=5`** creates backpressure: if a replica already has 5 in-flight requests, Serve queues the 6th until a slot opens.
- Seeing the same PID on all requests is normal in a local single-node setup where both replicas happen to share a process pool.


## Step 3 · Serving an sklearn Model via HTTP POST

The pattern for ML serving:
1. `__init__` — load the model **once** per replica (expensive, done at startup)
2. `__call__` — parse the request, run inference, return JSON

Loading in `__init__` means the model is hot in memory for every subsequent request.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np
import pickle

# ── Train a simple model and save it to bytes ────────────────────────────────
iris = load_iris()
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=200)),
])
clf.fit(iris.data, iris.target)
model_bytes = pickle.dumps(clf)  # serialize so the deployment can load it
print("Model trained. Pickle size:", len(model_bytes), "bytes")
print("Classes:", iris.target_names.tolist())


In [ ]:
import pickle as _pickle  # alias to avoid shadowing inside the class

@serve.deployment(num_replicas=1, ray_actor_options={"num_cpus": 0.5})
class IrisClassifier:
    """Serves predictions from an sklearn Pipeline over HTTP POST."""

    def __init__(self, model_bytes: bytes):
        # __init__ runs ONCE per replica at startup
        self.model = _pickle.loads(model_bytes)
        self.class_names = ["setosa", "versicolor", "virginica"]
        print("[IrisClassifier] Model loaded.")

    async def __call__(self, request):
        # Expected body: {"features": [[5.1, 3.5, 1.4, 0.2], ...]}
        body = await request.json()
        features = np.array(body["features"])
        preds = self.model.predict(features).tolist()
        proba = self.model.predict_proba(features).max(axis=1).tolist()
        return {
            "predictions": [self.class_names[p] for p in preds],
            "confidence":  [round(p, 4) for p in proba],
        }

# Pass model_bytes as a constructor argument via bind()
iris_handle = serve.run(
    IrisClassifier.bind(model_bytes),
    name="iris-app",
    route_prefix="/predict/iris",
)
time.sleep(1)

# Query with two Iris samples
sample = {"features": [[5.1, 3.5, 1.4, 0.2], [6.7, 3.0, 5.2, 2.3]]}
resp = requests.post("http://localhost:8000/predict/iris", json=sample)
print("Iris predictions:", resp.json())


### What just happened?
- **`bind(model_bytes)`** passes `model_bytes` as the first positional arg to `__init__` — this is how you inject pre-built artifacts into deployments.
- **`__init__` runs once per replica**, so `pickle.loads` only pays the deserialization cost at startup, not per-request.
- **`predict_proba`** is called on every request; the pipeline's `StandardScaler` step normalises inputs automatically.
- The `/predict/iris` route_prefix is independent of the class name — you control URL layout via `route_prefix`.


## Step 4 · Deployment Composition — Chaining Preprocessor → Predictor

Ray Serve lets you build **deployment graphs**: one deployment calls another using its async handle. This is the recommended pattern for multi-step pipelines because:
- Each stage scales independently (e.g., 1 Preprocessor, 3 Predictors)
- Stages can be reused across different application graphs

```
HTTP Request → Preprocessor.bind() → Predictor.bind()
                    ↓
               normalize, validate
                         ↓
                    run model, return JSON
```


In [ ]:
from ray.serve.handle import DeploymentHandle

@serve.deployment(num_replicas=1, ray_actor_options={"num_cpus": 0.25})
class Preprocessor:
    """Validates and normalises raw input before it reaches the model."""

    def __init__(self):
        # Simple per-feature mean/std computed from the training set
        self.mean = np.array([5.843, 3.057, 3.758, 1.199])
        self.std  = np.array([0.828, 0.436, 1.765, 0.763])

    async def __call__(self, raw_features: list) -> list:
        """Called by the Predictor via handle — NOT directly by HTTP."""
        arr = np.array(raw_features)
        if arr.ndim == 1:
            arr = arr.reshape(1, -1)
        normalised = ((arr - self.mean) / self.std).tolist()
        return normalised


@serve.deployment(num_replicas=1, ray_actor_options={"num_cpus": 0.5})
class ChainedPredictor:
    """HTTP ingress: receives requests, delegates preprocessing, runs model."""

    def __init__(self, preprocessor: DeploymentHandle, model_bytes: bytes):
        self.preprocessor = preprocessor          # handle to the Preprocessor deployment
        self.model = _pickle.loads(model_bytes)
        self.class_names = ["setosa", "versicolor", "virginica"]

    async def __call__(self, request):
        body = await request.json()
        raw = body["features"]

        # Call Preprocessor via its handle — async, non-blocking
        normalised = await self.preprocessor.remote(raw)

        # Run the model on the normalised features (skip the pipeline scaler here)
        arr = np.array(normalised)
        preds = self.model["lr"].predict(arr).tolist()  # direct LR, post-scale
        return {"predictions": [self.class_names[p] for p in preds]}


# Decompose the sklearn pipeline so we can access the LR step separately
lr_only = {"lr": clf.named_steps["lr"]}  # ChainedPredictor handles normalisation itself
lr_bytes = pickle.dumps(lr_only)

# Build the application graph — bind() nests deployments
preprocessor_node = Preprocessor.bind()
chained_app = ChainedPredictor.bind(preprocessor_node, lr_bytes)

chained_handle = serve.run(chained_app, name="chained-app", route_prefix="/chained")
time.sleep(1)

resp = requests.post("http://localhost:8000/chained", json={"features": [5.1, 3.5, 1.4, 0.2]})
print("Chained prediction:", resp.json())


### What just happened?
- **`Preprocessor.bind()`** creates a node; passing it into `ChainedPredictor.bind(preprocessor_node, ...)` tells Serve to wire a `DeploymentHandle` into `ChainedPredictor.__init__`.
- **`await self.preprocessor.remote(raw)`** calls the Preprocessor actor asynchronously — the ChainedPredictor does not block a thread while waiting.
- Each deployment in the graph **scales independently**: you can set `Preprocessor(num_replicas=1)` and `ChainedPredictor(num_replicas=3)` without changing any code.
- This graph pattern replaces the old `serve.ingress` + explicit handle lookups from older Serve versions.


## Step 5 · FastAPI Ingress with `serve.ingress`

For richer HTTP semantics (path params, query strings, request validation), wrap your deployment with a **FastAPI app** using the `@serve.ingress(app)` decorator.

| Approach | When to use |
|---|---|
| `async def __call__(self, request)` | Simple cases; raw Starlette Request |
| `@serve.ingress(fastapi_app)` | Need path params, Pydantic validation, OpenAPI docs |


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List

# ── FastAPI app definition (outside the class) ───────────────────────────────
app = FastAPI(title="Iris API", version="1.0")

class PredictRequest(BaseModel):
    features: List[List[float]]   # list of feature vectors

class PredictResponse(BaseModel):
    predictions: List[str]
    confidence: List[float]


@serve.deployment(num_replicas=1, ray_actor_options={"num_cpus": 0.5})
@serve.ingress(app)                        # <-- ties the FastAPI app to this deployment
class FastAPIIrisClassifier:
    """Serve deployment that exposes a typed FastAPI interface."""

    def __init__(self, model_bytes: bytes):
        self.model = _pickle.loads(model_bytes)
        self.class_names = ["setosa", "versicolor", "virginica"]

    @app.post("/predict", response_model=PredictResponse)
    async def predict(self, req: PredictRequest) -> PredictResponse:
        """POST /predict — returns class names and confidence scores."""
        arr = np.array(req.features)
        preds = self.model.predict(arr).tolist()
        proba = self.model.predict_proba(arr).max(axis=1).tolist()
        return PredictResponse(
            predictions=[self.class_names[p] for p in preds],
            confidence=[round(p, 4) for p in proba],
        )

    @app.get("/health")
    async def health(self):
        """GET /health — liveness probe for load balancers."""
        return {"status": "ok"}


fastapi_handle = serve.run(
    FastAPIIrisClassifier.bind(model_bytes),
    name="fastapi-iris",
    route_prefix="/v2",
)
time.sleep(1)

# Health check
health_resp = requests.get("http://localhost:8000/v2/health")
print("Health:", health_resp.json())

# Typed prediction
pred_resp = requests.post(
    "http://localhost:8000/v2/predict",
    json={"features": [[5.9, 3.0, 5.1, 1.8], [4.7, 3.2, 1.3, 0.2]]},
)
print("FastAPI predictions:", pred_resp.json())


### What just happened?
- **`@serve.ingress(app)`** patches the deployment class so that all FastAPI routes become methods of the actor, giving them access to `self` (and therefore `self.model`).
- **Pydantic models** (`PredictRequest`, `PredictResponse`) give you automatic request validation and a free `/v2/docs` OpenAPI page.
- **`/health`** is a standard liveness probe — production load balancers ping this to decide whether to route traffic to a replica.
- The `route_prefix="/v2"` prefix is prepended to all FastAPI routes: `/v2/predict`, `/v2/health`, `/v2/docs`.


## Step 6 · Autoscaling Configuration

Instead of a fixed `num_replicas`, Serve can autoscale based on request load:

```python
@serve.deployment(
    autoscaling_config={
        "min_replicas": 1,
        "max_replicas": 8,
        "target_num_ongoing_requests_per_replica": 10,
        "upscale_delay_s": 5,      # wait 5s of sustained load before adding replicas
        "downscale_delay_s": 60,   # wait 60s of low load before removing replicas
    }
)
class AutoScaledService: ...
```

In a notebook we demonstrate the configuration but do not run it (autoscaling needs a multi-second load test to trigger).


In [ ]:
# Inspect currently running deployments
status = serve.status()
print("Running applications:")
for app_name, app_status in status.applications.items():
    print(f"  {app_name}: {app_status.status}")
    for dep_name, dep_status in app_status.deployments.items():
        print(f"    └─ {dep_name}: {dep_status.status}")


### What just happened?
- **`serve.status()`** returns a live snapshot of every application and deployment the controller knows about.
- Each deployment reports a status string: `HEALTHY`, `UPDATING`, or `UNHEALTHY`.
- This is the programmatic equivalent of the **Ray Dashboard → Serve** tab.


In [ ]:
# Challenge: Build a sentiment scoring deployment
#
# Requirements:
#   1. Define a @serve.deployment class called SentimentScorer
#   2. In __init__, create a simple word-count based scorer:
#      positive_words = {"great", "good", "excellent", "happy", "love"}
#      negative_words = {"bad", "terrible", "awful", "hate", "poor"}
#   3. In __call__, accept {"text": "..."} via HTTP POST
#      - count positive and negative word matches
#      - return {"text": ..., "score": (pos - neg), "sentiment": "positive"/"negative"/"neutral"}
#   4. Deploy it at route_prefix="/sentiment"
#   5. Test with: requests.post("http://localhost:8000/sentiment", json={"text": "This is great and excellent"})

# Your solution here:
# @serve.deployment(...)
# class SentimentScorer:
#     def __init__(self): ...
#     async def __call__(self, request): ...


In [ ]:
# Shutdown Serve cleanly (releases all actor resources)
serve.shutdown()
ray.shutdown()
print("Ray Serve shut down.")


---
## Day 7 key concepts recap

| Concept | What to remember |
|---|---|
| `@serve.deployment` | Registers a class with Serve; does not instantiate yet |
| `.bind()` | Creates an immutable application graph node |
| `serve.run()` | Materialises the graph — spawns actors, registers routes |
| `num_replicas` | Static replica count; use with `max_concurrent_queries` for model serving |
| `ray_actor_options` | Per-replica resource reservations — same syntax as `@ray.remote` |
| Deployment graph | Chain deployments by passing `Foo.bind()` into `Bar.bind(foo_node)` |
| `serve.ingress(app)` | Attaches a FastAPI app to a deployment for typed routes and Pydantic validation |
| `serve.status()` | Programmatic health check across all deployments |

> **Tip:** Ray Serve deployments are actors under the hood. Setting `num_replicas=N` creates N actor instances behind a load balancer. For large models, keep `num_replicas` low and increase `max_concurrent_queries` instead.

---
## What's next
**Day 8** → Ray Workflows and Job Submission — package your pipeline as a durable, checkpointed workflow and submit it to a Ray cluster via the Jobs API.

Mark Day 7 complete in your [tracker](../index.html).
